In [24]:
# # !pip uninstall numpy -y
# !pip install numpy==1.24.4

In [25]:
import cv2
import os
import numpy as np
import mediapipe as mp
import face_recognition
from ultralytics import YOLO
from mediapipe.python.solutions import pose

In [26]:
person_model = YOLO("yolov8s.pt")
weapon_model = YOLO (r"runs\detect\weapon_model\weights\best.pt")

In [27]:
mp_pose = mp.solutions.pose
pose = mp_pose.Pose(
    static_image_mode=False,
    model_complexity=1,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)


In [28]:
# for file in os.listdir("known_faces"):
#     image = face_recognition.load_image_file(f"known_faces/{file}")
#     encodings = face_recognition.face_encodings(image)
#     if encodings:
#         known_face_encodings.append(encodings[0])
#         name = os.path.splitext(file)[0]
#         known_face_names.append(name)

# print("Loaded Known Faces:", known_face_names)

known_face_encodings = []
known_face_names = []

for file in os.listdir("known_faces"):
    
    # Only process image files
    if file.lower().endswith((".jpg", ".jpeg", ".png")):
        
        image_path = os.path.join("known_faces", file)
        image = face_recognition.load_image_file(image_path)
        encodings = face_recognition.face_encodings(image)
        
        if encodings:
            known_face_encodings.append(encodings[0])
            name = os.path.splitext(file)[0]
            known_face_names.append(name)

print("Loaded Known Faces:", known_face_names)

Loaded Known Faces: []


In [29]:
armed_ids = set()
identity_map = {}

In [30]:
def detect_posture(landmarks):

    left_shoulder = landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value]
    right_shoulder = landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value]
    left_hip = landmarks[mp_pose.PoseLandmark.LEFT_HIP.value]
    right_hip = landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value]
    left_wrist = landmarks[mp_pose.PoseLandmark.LEFT_WRIST.value]
    right_wrist = landmarks[mp_pose.PoseLandmark.RIGHT_WRIST.value]

    shoulder_y = (left_shoulder.y + right_shoulder.y) / 2
    hip_y = (left_hip.y + right_hip.y) / 2

    if abs(shoulder_y - hip_y) < 0.08:
        return "FALLEN"

    if left_wrist.y < left_shoulder.y and right_wrist.y < right_shoulder.y:
        return "HANDS UP"

    return "STANDING"

In [31]:
video_path = "Prediction1.avi"
cap = cv2.VideoCapture(video_path)

In [32]:
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter(
    "output_face.mp4",
    fourcc,
    fps,
    (frame_width, frame_height)
)

In [33]:
# cv2.namedWindow("Smart AI Surveillance System", cv2.WINDOW_NORMAL)
# cv2.resizeWindow("Smart AI Surveillance System", 1000, 700)




In [34]:


while True:
    ret, frame = cap.read()
    if not ret:
        break

    # ---- PERSON TRACKING ----
    person_results = person_model.track(
        frame,
        persist=True,
        conf=0.4,
        classes=[0],
        verbose=False
    )

    # ---- WEAPON DETECTION ----
    weapon_results = weapon_model(frame, conf=0.55, verbose=False)

    persons = []
    weapons = []

    # ---------------------------
    # COLLECT PERSONS
    # ---------------------------
    if person_results[0].boxes.id is not None:
        for box, track_id in zip(
                person_results[0].boxes.xyxy,
                person_results[0].boxes.id):

            x1, y1, x2, y2 = map(int, box)
            track_id = int(track_id)
            persons.append((track_id, x1, y1, x2, y2))

    # ---------------------------
    # COLLECT WEAPONS
    # ---------------------------
    for box in weapon_results[0].boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        weapons.append((x1, y1, x2, y2))

    # ---------------------------
    # ASSOCIATE WEAPON TO PERSON
    # ---------------------------
    for wx1, wy1, wx2, wy2 in weapons:
        cx = (wx1 + wx2) // 2
        cy = (wy1 + wy2) // 2

        for track_id, px1, py1, px2, py2 in persons:
            if px1 <= cx <= px2 and py1 <= cy <= py2:
                armed_ids.add(track_id)

    # ---------------------------
    # PROCESS EACH PERSON
    # ---------------------------
    for track_id, px1, py1, px2, py2 in persons:

        person_roi = frame[py1:py2, px1:px2]

        # ===============================
        # FACE RECOGNITION
        # ===============================

        if track_id not in identity_map:
            name = "Unknown"

            if person_roi.size > 0:
                rgb_roi = cv2.cvtColor(person_roi, cv2.COLOR_BGR2RGB)
                face_locations = face_recognition.face_locations(rgb_roi)
                face_encodings = face_recognition.face_encodings(rgb_roi, face_locations)

                for face_encoding in face_encodings:
                    matches = face_recognition.compare_faces(
                        known_face_encodings, face_encoding)
                    face_distances = face_recognition.face_distance(
                        known_face_encodings, face_encoding)

                    if len(face_distances) > 0:
                        best_match_index = np.argmin(face_distances)
                        if matches[best_match_index]:
                            name = known_face_names[best_match_index]

            identity_map[track_id] = name
        else:
            name = identity_map[track_id]

        # ===============================
        # POSE DETECTION
        # ===============================

        posture = "UNKNOWN"

        if person_roi.size > 0:
            roi_rgb = cv2.cvtColor(person_roi, cv2.COLOR_BGR2RGB)
            results = pose.process(roi_rgb)

            if results.pose_landmarks:
                posture = detect_posture(results.pose_landmarks.landmark)

        # ===============================
        # COLOR LOGIC
        # ===============================

        if track_id in armed_ids:
            color = (0, 0, 255)  # RED
            status = "ARMED"
        else:
            color = (0, 255, 0)  # GREEN
            status = "NORMAL"

        # ===============================
        # DRAW
        # ===============================

        cv2.rectangle(frame, (px1, py1), (px2, py2), color, 2)

        label = f"ID {track_id} | {name} | {posture} | {status}"

        cv2.putText(frame, label,
                    (px1, py1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.5,
                    color,
                    2)
    out.write(frame)

    display_frame = cv2.resize(frame, (1000, 700))
    cv2.imshow("Smart AI Surveillance System", display_frame)
    # cv2.imshow("Smart AI Surveillance System", frame)
    # cv2.namedWindow("Smart AI Surveillance System", cv2.WINDOW_NORMAL)
    # cv2.resizeWindow("Smart AI Surveillance System", 1000, 700)

    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
out.release()
cv2.destroyAllWindows()
